In [ ]:
"""실습 2 정답: 가열로 존 편차와 소재 온도·라인 속도를 비교합니다."""

from pathlib import Path

import pandas as pd



DATA = "..//3일차_실습//03-01_유압·열설비_신호_가열로온도.csv"
TAG_FILE = ".//3일차_실습//03-01_유압·열설비_신호_계통태그목록.csv"
REQUIRED_COLUMNS = {
    "date",
    "FUR01_Z1_TEMP_L",
    "FUR01_Z1_TEMP_R",
    "FUR01_Z2_TEMP_L",
    "FUR01_Z2_TEMP_R",
    "FUR01_Z3_TEMP_L",
    "FUR01_Z3_TEMP_R",
    "FUR01_MAT_TEMP",
    "FUR01_LINE_SPEED",
}

for path in (DATA, TAG_FILE):
    if not path.is_file():
        raise FileNotFoundError(f"필요한 실습 파일을 찾을 수 없습니다: {path}")

df = pd.read_csv(DATA, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
missing = REQUIRED_COLUMNS.difference(df.columns)
if missing:
    raise ValueError(f"가열로 데이터에 필수 컬럼이 없습니다: {sorted(missing)}")
if len(df) < 60:
    raise ValueError(f"60일 비교에 필요한 행이 부족합니다: {len(df)}행")

for zone in ("Z1", "Z2", "Z3"):
    df[f"{zone}_diff_c"] = (
        df[f"FUR01_{zone}_TEMP_R"] - df[f"FUR01_{zone}_TEMP_L"]
    )

first_20_means = df.iloc[:20][[f"{zone}_diff_c" for zone in ("Z1", "Z2", "Z3")]].mean()
print("=== Step 1: 처음 20일 평균 좌우 편차 (°C) ===")
print(first_20_means.rename(index=lambda col: col.removesuffix("_diff_c")).round(1))

day_positions = [0, 19, 39, 59]
day_labels = ["1일차", "20일차", "40일차", "60일차"]
days = df.iloc[day_positions].copy()
days.insert(0, "시점", day_labels)
day_diffs = days[
    ["시점", "Z1_diff_c", "Z2_diff_c", "Z3_diff_c"]
].rename(
    columns={
        "Z1_diff_c": "Z1 편차(°C)",
        "Z2_diff_c": "Z2 편차(°C)",
        "Z3_diff_c": "Z3 편차(°C)",
    }
)
print("\n=== Step 2: 시점별 좌우 편차 (°C) ===")
print(day_diffs.to_string(index=False))

increase = days.iloc[-1][[f"{z}_diff_c" for z in ("Z1", "Z2", "Z3")]] - days.iloc[0][
    [f"{z}_diff_c" for z in ("Z1", "Z2", "Z3")]
]
largest_zone = increase.idxmax().removesuffix("_diff_c")
print(
    f"1~60일차 편차 증가가 가장 큰 존: {largest_zone} "
    f"({increase.max():.1f} °C 증가)"
)

z2 = days[
    ["시점", "FUR01_Z2_TEMP_L", "FUR01_Z2_TEMP_R"]
].rename(
    columns={
        "FUR01_Z2_TEMP_L": "좌측 온도(°C)",
        "FUR01_Z2_TEMP_R": "우측 온도(°C)",
    }
)
print("\n=== Step 3: Z2 좌우 온도 (°C) ===")
print(z2.to_string(index=False))
print("우측 온도는 1228 °C로 유지되고 좌측 온도는 하락합니다.")

speed_positions = [0, 19, 43, 44, 59]
speed_days = ["1일차", "20일차", "44일차", "45일차", "60일차"]
process = df.iloc[speed_positions][
    ["FUR01_MAT_TEMP", "FUR01_LINE_SPEED"]
].copy()
process.insert(0, "시점", speed_days)
process = process.rename(
    columns={
        "FUR01_MAT_TEMP": "소재 온도(°C)",
        "FUR01_LINE_SPEED": "라인 속도(m/min)",
    }
)
print("\n=== Step 4: 소재 온도와 라인 속도 ===")
print(process.to_string(index=False))

day44 = df.iloc[43]
day45 = df.iloc[44]
print(
    "\n44→45일차: 라인 속도 "
    f"{day44['FUR01_LINE_SPEED']:.0f}→{day45['FUR01_LINE_SPEED']:.0f} m/min, "
    f"소재 온도 {day44['FUR01_MAT_TEMP']:.0f}→"
    f"{day45['FUR01_MAT_TEMP']:.0f} °C."
)
print(
    "\n=== Step 5: 해석 ===\n"
    "Z2는 좌측 열전대 온도 하락과 함께 좌우 편차가 확대됩니다. "
    "좌측 버너·연료/공기 공급·제어 출력과 열전대/배선을 대조 점검할 단서입니다.\n"
    "45일차에는 라인 속도 증가와 소재 온도 하락이 함께 나타나 체류 시간 감소가 "
    "가능성 있는 설명이지만, 이 데이터만으로 인과관계를 확정할 수는 없습니다.\n"
    "버너·연소·존 설정, 소재 조건, 운전/알람 이력 및 센서 교정 상태를 확인하세요. "
    "교육용 데이터의 추세를 실제 경보값이나 고장 판정으로 사용하지 않습니다."
)
